# GaLS elasticity equations and attributed MSL comparisons

This standalone calculation refines the same unit-square trigonometric physical
problem as the retained MSL GaLS comparison: shear modulus 1, Lamé parameter
4999, zero boundary displacement and pressure amplitude one. The current P1/P1
locals use a uniform triangle macro mesh and a linear vector trace with four
fine edges per face. This mesh differs from the archived crisscross comparison;
the errors below compare against the independently defined analytical fields.

Herrmann pressure is $p=-\lambda\,\operatorname{div}u$ and Cauchy stress is
$\sigma=2\mu\,\varepsilon(u)-pI$. The skeleton represents $-\sigma n$.
The seven original MSL acquisitions retain their revisions and scalar norms.
Their external coefficient files are optional historical replay inputs, and are
not distributed with the Python package. See [the attributed comparison](https://ipes-lncc.github.io/pymhm/cases/elasticity-reference/).

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/74e0250eaa4d7032fca42878eb8312d756673045a9467a4e86dad784dc97087c/15_elasticity_reference-companion.zip"
COMPANION_SHA256 = "74e0250eaa4d7032fca42878eb8312d756673045a9467a4e86dad784dc97087c"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("elasticity/15_elasticity_reference.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


## Local forms, physical kernel and pressure identity

The element provider declares displacement and pressure blocks, the negative
GaLS residual term and the three globally centered rigid motions. For piecewise
linear displacement and constant shear the fine-cell residual is $R(u,p)=-\nabla p$.
The parameter is $\alpha=1/(4\mu)$ with the matching-mesh fine-edge condition.
Boundary displacement enters the global weak traction equation. The pressure
identity uses the physical compliance weights rather than an arbitrary pressure
shift. Inspecting the provider reveals each actual block used by assembly.

In [ ]:
from functools import partial
import inspect
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from examples.elasticity_data import TrigonometricElasticityData
from examples.formulations.elasticity import (
    DisplacementPressureSpace, displacement_pressure_equations, displacement_pressure_fields,
)
from pymhm import Equation, MultiscaleProblem, TriangleMesh, FaceSpace, SkeletonSpace, assemble
from pymhm.fem.scalar.operators import boundary_data

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
print(inspect.getsource(displacement_pressure_equations))
exact = TrigonometricElasticityData(lame_lambda=4999)
current = []
for n in (1, 2, 4):
    mesh = TriangleMesh.unit_square(n)
    skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces), 2)
    space = DisplacementPressureSpace(mesh, skeleton, 4, 8)
    provider = partial(displacement_pressure_equations, space=space, source=exact.source,
                       lame_lambda=exact.lame_lambda)
    boundary, fixed = boundary_data(skeleton, (0.0, 0.0), order=8)
    problem = MultiscaleProblem(
        Equation(0, -np.r_[boundary, np.zeros(3*len(mesh.cells))]), provider,
        range(len(mesh.cells)), skeleton.size, (3,)*len(mesh.cells), fixed=fixed,
    )
    system = assemble(problem)
    compliance_scale = max(record[6] for record in system.local_metadata)
    pressure_identity = system.mean_constraint([record[5]/compliance_scale for record in system.local_metadata], 0.0)
    coefficients = system.solve(constraints=(pressure_identity,))
    result = displacement_pressure_fields(system, coefficients, space, exact.lame_lambda)
    current.append((n, result.l2_error(exact.displacement, 10),
                    result.pressure_l2_error(exact.pressure, 10),
                    result.stress_l2_error(exact.stress, 10), coefficients.raw_residual))
print("Current analytical errors (n, displacement, pressure, stress, equation residual):")
for row in current:
    print(row)
fig, ax = plt.subplots(figsize=(6.5, 4))
for index, label in ((1, "Displacement"), (2, "Pressure"), (3, "Cauchy stress")):
    ax.loglog([1/row[0] for row in current], [row[index] for row in current], "o-", label=label)
ax.set(xlabel="Macro mesh size", ylabel="Physical L2 error")
ax.legend(); ax.grid(True, which="both"); fig.tight_layout()
plt.show()
record = json.loads((ROOT / "examples/results/elasticity-reference.json").read_text())
print("Retained original reference:", record["reference"])
print("Retained original revisions:", record["revisions"])


## Optional original MSL coefficient replay

The following section checks and displays the original matched crisscross,
P2/P2 and P3/P3 acquisitions when `python -m scripts.run_notebooks /path/to/15_elasticity_reference.ipynb --historical` is selected and
the checksum-matching external archives are available. It is separate from the
current analytical refinement above.

In [ ]:
if HISTORICAL_REPLAY:
    from pathlib import Path
    import hashlib
    import json
    import numpy as np
    from IPython.display import SVG, display
    
    record_path = root / "examples/results/elasticity-reference.json"
    record = json.loads(record_path.read_text())
    rows = record["rows"]
    assert len(rows) == 7
    for row in rows:
        archive = record_path.parent / row["archive"]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
        with np.load(archive, allow_pickle=False) as arrays:
            for key in arrays.files:
                assert np.all(np.isfinite(arrays[key])), (archive.name, key)
            assert int(arrays["degree"]) == row["degree"]
            assert len(arrays["macro_cells"]) == row["macro_triangles"]
    print(record["reference"])
    print(record["revisions"])
    print("Seven archived field checksums and array contracts passed.")
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks elasticity/15_elasticity_reference.ipynb --historical with the attributed archives.")


## Agreement and accuracy are different diagnostics

The stored `difference_l2` values compare both reconstructed polynomial fields
on the same physical quadrature. They include displacement, its full broken
gradient, pressure and full Cauchy stress. No pressure-mean shift was applied.

The checks below bound the maximum recorded implementation difference and
verify the reverse triangle inequality against the two approximation errors.
They inspect recorded norms; their scale is not a discretization convergence
rate, and agreement between codes does not by itself establish exact-solution
accuracy.


In [ ]:
if HISTORICAL_REPLAY:
    limits = {"u_l2": 3e-13, "p_l2": 9e-12,
              "u_h1_semi": 2e-11, "stress_l2": 4e-11}
    maxima = {metric: max(row["difference_l2"][metric] for row in rows) for metric in limits}
    for metric, limit in limits.items():
        assert maxima[metric] <= limit
        for row in rows:
            first, second = row["msl"][metric], row["pymhm"][metric]
            rounding = 64*np.finfo(float).eps*(1+abs(first)+abs(second))
            assert abs(first-second) <= row["difference_l2"][metric] + rounding
    maxima
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks elasticity/15_elasticity_reference.ipynb --historical with the attributed archives.")


In [ ]:
if HISTORICAL_REPLAY:
    display(SVG(filename=str(root / "docs/figures/elasticity-reference/field-differences.svg")))
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks elasticity/15_elasticity_reference.ipynb --historical with the attributed archives.")


## Five matched mesh levels and two additional local pairs

A crisscross grid with `n` Cartesian squares per side has `4*n**2` macrotriangles
and macro diameter `H=1/n`. P1/P1 uses four fine subdivisions per macro edge and
stabilization `alpha=.1`, at `n=1,2,4,8,16`. The P2/P2 and P3/P3 checks both use
`n=4` and `alpha=.001`, with local refinements two and one respectively.

Assembly requested quadrature order 10, and field comparison order 12. Each
implementation may realize its assembly order using a different quadrature
rule. The additional-degree checks change the fine mesh too, so they are not
a fixed-mesh polynomial-rate experiment.


In [ ]:
if HISTORICAL_REPLAY:
    for row in rows:
        print(f"n={row['macro_resolution']:2}, degree={row['degree']}, "
              f"local refinement={row['local_refinement']}: {row['pymhm']}")
    display(SVG(filename=str(root / "docs/figures/elasticity-reference/convergence.svg")))
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks elasticity/15_elasticity_reference.ipynb --historical with the attributed archives.")


## Exact, MSL and pyMHM fields on the actual macro mesh

The maps show the P3/P3 case at `n=4`. The analytical, MSL and pyMHM fields share
one scale. The two approximation-error maps share another scale, while the
implementation difference has its own labeled colorbar. Macro boundaries appear
in every panel. Each broken fine-element polynomial is evaluated separately;
one-sided stress and pressure values are not smoothed across interfaces.

The displacement-pressure GaLS stress is symmetric. This reconstruction does
not enforce global H(div) conformity of stress.


In [ ]:
if HISTORICAL_REPLAY:
    for field in ("u1", "pressure", "stress11"):
        display(SVG(filename=str(root / f"docs/figures/elasticity-reference/{field}.svg")))
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks elasticity/15_elasticity_reference.ipynb --historical with the attributed archives.")


## Recorded reference provenance

The report identifies the exact MSL_MHM, MSL_CG and MSL_CORE revisions and the
mixed-field adapter digests. Native GaLS element assembly was retained. The
adapter pads rigid-mode source projection with zero pressure rows and
reconstructs pressure using the same native skeletal coefficients as
displacement. This is a recorded adapter around the native operators, not an
assertion that an unchanged historical application generated the files.

The consistent amplitude-one benchmark matches the MSL input. It does not mix
the incompatible coefficients printed for displacement, pressure and forcing
in the article. The full derivation and scope are in
`docs/cases/elasticity.md` and `docs/cases/elasticity-reference.md`.

To redraw these archived fields, run
`python -m examples.plot_elasticity_reference`.
